# 20260725_EDA_구조환경지표_21개_검증본_interim_생성
- 작성자: 이정연
- 이슈 #38 참고


## 1. 실행 환경 설정


In [1]:
import sys
from pathlib import Path

import pandas as pd

repo_root = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / ".git").exists()), None
)
if repo_root is None:
    raise FileNotFoundError("현재 실행 위치의 상위 경로에서 Git 저장소를 찾지 못했습니다.")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"저장소 루트: {repo_root}")

저장소 루트: /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha


## 1-1. 공통 설정 - 지역명 매핑


In [2]:
lookup_path = repo_root / "data" / "lookup" / "시도_지역코드_매핑.csv"
region_lookup = pd.read_csv(lookup_path)

region_order = ["전국", *region_lookup["지역"].tolist()]

LEGACY_REGION_NAMES = {
    "강원도": "강원",
    "전라북도": "전북",
    "제주도": "제주",
}
NATIONWIDE_ALIASES = {"전체": "전국", "계": "전국"}

region_map = (
    {r: r for r in region_order}
    | dict(zip(region_lookup["지역명_전체"], region_lookup["지역"]))
    | LEGACY_REGION_NAMES
    | NATIONWIDE_ALIASES
)

print(f"region_order: {len(region_order)}개 지역")

region_order: 18개 지역


## 2. 재정팀 통합 파일 로드 - v1과 v2 대조

`20260725_EDA_구조환경지표_통합파일_결측치_검증.ipynb`에서 21개 지표 전부 v1 통합 파일이
개별 가공 파일과 정확히 일치함을 검증했다. 그런데 이후 재정팀이 v2로 파일을 업데이트했다는
연락을 받아, 여기서는 v2를 기준으로 interim 데이터셋을 만든다. 먼저 v1과 v2 중 어디가
바뀌었는지 전수 비교한다.


In [3]:
measures_dir = repo_root / "data" / "raw" / "지표별_측정값"
v1_path = measures_dir / "구조환경지표 측정값_합본_30개 중 21개.xlsx"
v2_path = measures_dir / "구조환경지표 측정값_합본_30개 중 21개_v2.xlsx"

merged_v1 = pd.read_excel(v1_path, sheet_name="Sheet1")
merged_v1.columns = [c if isinstance(c, int) else str(c).strip() for c in merged_v1.columns]

merged_df = pd.read_excel(v2_path, sheet_name="세부지표별 측정값")
merged_df.columns = [c if isinstance(c, int) else str(c).strip() for c in merged_df.columns]

print(f"v1: {v1_path.relative_to(repo_root)}")
print(f"v2: {v2_path.relative_to(repo_root)}")
print(f"v2 크기: {merged_df.shape[0]}행 x {merged_df.shape[1]}열")

years_all = [c for c in merged_df.columns if isinstance(c, int)]
changed_indicators = []
for label in merged_df["세부지표"].unique():
    a = merged_v1.loc[merged_v1["세부지표"] == label, ["지역", *years_all]].set_index("지역")
    b = merged_df.loc[merged_df["세부지표"] == label, ["지역", *years_all]].set_index("지역")
    for col in years_all:
        a[col] = pd.to_numeric(a[col], errors="coerce")
        b[col] = pd.to_numeric(b[col], errors="coerce")
    if not a.reindex_like(b).round(6).equals(b.round(6)):
        changed_indicators.append(label)

print("\nv1과 v2가 다른 세부지표:", changed_indicators)

v1: data/raw/지표별_측정값/구조환경지표 측정값_합본_30개 중 21개.xlsx
v2: data/raw/지표별_측정값/구조환경지표 측정값_합본_30개 중 21개_v2.xlsx
v2 크기: 377행 x 15열

v1과 v2가 다른 세부지표: ['보육시설 보급률', '산후조리원 이용 요금', '결혼에 대한 인식']


## 3. 저희 독립 재현값이 v2와 일치하는지 확인

v1→v2에서 바뀐 3개 지표(보육시설 보급률, 산후조리원 이용 요금, 결혼에 대한 인식)에 대해
저희가 원자료로 독립 재현한 값(`20260724_EDA_구조환경지표_21개_원자료기반_전수검증.ipynb`
7·13·20절에서 생성해 `data/interim/`에 저장해둔 값)과 v2를 대조한다. 재정팀의 실제 수정과
저희 재현 로직이 서로 다른 경로로 같은 결론에 도달했는지 확인하는 교차검증이다.


In [4]:
interim_dir = repo_root / "data" / "interim" / "구조환경지표_검증"

cross_check_files = {
    "보육시설 보급률": "보육시설_보급률_정원기준_수정값.csv",
    "산후조리원 이용 요금": "산후조리원_이용요금_특실대체_수정값.csv",
    "결혼에 대한 인식": "결혼에_대한_인식_원자료재현_수정값.csv",
}

for label, file_name in cross_check_files.items():
    ours = pd.read_csv(interim_dir / file_name).set_index("지역")
    ours.columns = [int(c) for c in ours.columns]
    v2_slice = merged_df.loc[merged_df["세부지표"] == label].set_index("지역")[ours.columns]
    for col in ours.columns:
        ours[col] = pd.to_numeric(ours[col], errors="coerce")
        v2_slice[col] = pd.to_numeric(v2_slice[col], errors="coerce")
    max_diff = (ours.reindex(region_order) - v2_slice.reindex(region_order)).abs().max().max()
    print(f"{label}: 저희 재현값 vs v2 최대오차 = {max_diff:.2e}")

보육시설 보급률: 저희 재현값 vs v2 최대오차 = 1.42e-14
산후조리원 이용 요금: 저희 재현값 vs v2 최대오차 = 5.68e-14
결혼에 대한 인식: 저희 재현값 vs v2 최대오차 = 4.44e-16


## 4. 검증상태 표시 후 저장

v2 통합 파일을 그대로 기준으로 쓰되, `검증상태` 컬럼에 지표별로 실제 검증 수준을 구분해서 남긴다.
21개 지표를 뭉뚱그려 "전부 오차 0으로 재현됨"이라고 하면 사실과 다르다 - 실제로는 4가지로 나뉜다.

1. **v2에서 수정 확인됨**(3개): 보육시설 보급률, 산후조리원 이용 요금, 결혼에 대한 인식
2. **원자료 재현 검증 완료, 오차 0**(13개): 나머지 대부분
3. **원자료 재현했으나 불일치 남음, 원인 미상**(2개): 가족친화인증기업 비율, 도시공원 보급도
4. **공표값 그대로 사용, 라벨만 확인**(3개): 소득수준, 사교육비 지출액, 문화기반시설 보급도


In [5]:
final_df = merged_df.copy()

# 지표별 실제 검증 상태를 반영한다(21개 지표를 뭉뚱그려 "전부 오차 0"이라고 하지 않는다).
CORRECTED_NOTES = {
    "보육시설 보급률": (
        "v2에서 수정됨 - 정원 기준으로 재계산(v1은 어린이집 수 기준 오류). "
        "저희 독립 재현값과 오차 0으로 일치 확인"
    ),
    "산후조리원 이용 요금": (
        "v2에서 수정됨 - 일반실 없으면 특실 대체 적용(v1은 결측 시설 제외). "
        "저희 독립 재현값과 오차 0으로 일치 확인"
    ),
    "결혼에 대한 인식": (
        "v2에서 수정됨 - 원데이터(%) 기반 가중평균으로 재계산(v1은 반올림이 아닌 실제 오류). "
        "저희 독립 재현값과 오차 0으로 일치 확인"
    ),
}

UNRESOLVED_NOTES = {
    "가족친화인증기업 비율": (
        "원자료 재현 완료했으나 2024년 대구·광주만 원자료 직접 카운트와 불일치(79건) - "
        "재정팀도 원인 특정 못 함, 원인 미상으로 남아있음"
    ),
    "도시공원 보급도": (
        "원자료가 로컬에 없어 라벨(정의)만 확인함 - 일부 지역·연도에서 30~100% 급등락 발견, "
        "원인은 KOSIS 원 통계 자체의 특성으로 추정되나 확인 못 함"
    ),
}

LABEL_ONLY_NOTES = {
    "소득수준": "공표값을 그대로 사용하는 지표 - 원자료가 로컬에 없어 라벨(정의)만 확인함, 값 재현 검증 대상 아님",
    "사교육비 지출액": "공표값을 그대로 사용하는 지표 - 원자료가 로컬에 없어 라벨(정의)만 확인함, 값 재현 검증 대상 아님",
    "문화기반시설 보급도": "공표값을 그대로 사용하는 지표 - 원자료가 로컬에 없어 라벨(정의)만 확인함, 값 재현 검증 대상 아님",
}

final_df["검증상태"] = "원자료 재현 검증 완료(오차 0)"
for label, note in {**CORRECTED_NOTES, **UNRESOLVED_NOTES, **LABEL_ONLY_NOTES}.items():
    final_df.loc[final_df["세부지표"] == label, "검증상태"] = note

output_path = interim_dir / "구조환경지표_21개_검증본.csv"
final_df.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"저장 완료: {output_path.relative_to(repo_root)}")
print(f"크기: {final_df.shape[0]}행 x {final_df.shape[1]}열")
display(final_df["검증상태"].value_counts())

저장 완료: data/interim/구조환경지표_검증/구조환경지표_21개_검증본.csv
크기: 377행 x 16열


검증상태
원자료 재현 검증 완료(오차 0)                                                                          233
공표값을 그대로 사용하는 지표 - 원자료가 로컬에 없어 라벨(정의)만 확인함, 값 재현 검증 대상 아님                                    54
v2에서 수정됨 - 정원 기준으로 재계산(v1은 어린이집 수 기준 오류). 저희 독립 재현값과 오차 0으로 일치 확인                            18
원자료가 로컬에 없어 라벨(정의)만 확인함 - 일부 지역·연도에서 30~100% 급등락 발견, 원인은 KOSIS 원 통계 자체의 특성으로 추정되나 확인 못 함     18
v2에서 수정됨 - 일반실 없으면 특실 대체 적용(v1은 결측 시설 제외). 저희 독립 재현값과 오차 0으로 일치 확인                           18
원자료 재현 완료했으나 2024년 대구·광주만 원자료 직접 카운트와 불일치(79건) - 재정팀도 원인 특정 못 함, 원인 미상으로 남아있음                18
v2에서 수정됨 - 원데이터(%) 기반 가중평균으로 재계산(v1은 반올림이 아닌 실제 오류). 저희 독립 재현값과 오차 0으로 일치 확인                 18
Name: count, dtype: int64